# Phase 6 submission

Prep v2 artifacts, TWO configs x five folds, probability mean over all ten models.

The single variable against the scored 0.905 is a *second config* -- 0.905 was already a
five-fold average, so this does not test "ensembling", it tests config diversity on top of
fold averaging. Phase 6 measured c2+c3 at +0.0108 over c2 on fold-0 val, but that was over
single-fold members while the confirm separately showed fold-averaging buying +0.0095 on
gold, so the two effects may overlap.

Members differ in backbone, head AND slot count off ONE shared prep pass -- the artifact is
written once per study and each member reads it with its own slot subset and read size.

Three things differ from `phase5-submit`, all of them consequences of the artifact change:

- **`prep_slots`, not `prep_study`.** Test studies must reach the model through the same
  130mm physical-scale crop, six-slot layout and contiguous 3-slice groups the training
  artifacts were built with. Prepping at 224 directly would be faster and wrong: training
  saw 336 stored then resized to 224, and that resampling chain is part of what the model
  learned.
- **The loader returns a presence mask**, and it is passed to the model. Dropping it would
  make every absent slot read as real black anatomy, which is exactly the bias the masked
  pooling exists to remove -- and it would show up only as a slightly worse score.
- **`pretrained=False`.** Internet is off and every weight is overwritten by the checkpoint
  anyway; leaving it True would try to reach the network and fail the submission outright.

The 0.5 fallback is unchanged and remains non-negotiable: a submission that raises on one
hidden study scores zero.


In [ ]:
import glob, os, shutil, sys, tempfile, time

GIT_SHA = 'phase7-submit'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
# name, checkpoint pattern, backbone, head, how many of the six slots it reads, read size.
# c3 reads slots[:2] and a mean pool; c2 reads all six through the attention head. Both were
# trained on the SAME prep v2 artifact, which is why one prep pass can serve both.
MEMBERS = [
    dict(name='c2', pattern='c2_6slot_attn_fold*.pt', backbone='efficientnet_b0',
         head='slot_attention', n_slots=6, read=224),
    dict(name='c3', pattern='c3_dinov2_2slot_fold*.pt',
         backbone='vit_small_patch14_dinov2.lvd142m', head='mean', n_slots=2, read=224),
]
for _m in MEMBERS:
    _m['ckpts'] = sorted(glob.glob(f"/kaggle/input/**/{_m['pattern']}", recursive=True))
    assert len(_m['ckpts']) == 5, (
        f"{_m['name']}: expected 5 fold checkpoints matching {_m['pattern']}, "
        f"found {len(_m['ckpts'])}: {_m['ckpts']}")
CKPTS = [c for _m in MEMBERS for c in _m['ckpts']]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

# A stale src here does not crash -- it silently prepares different pixels from the
# ones the checkpoints were trained on, and the only symptom is a lower score.
import inspect
from knee.dicom import SLOTS, select_slots, select_slice_groups
from knee.model import KneeModel
from knee.prep import crop_to_mm, prep_slots
assert 'head' in inspect.signature(KneeModel.__init__).parameters, 'src predates the c2 head'
# The artifact these checkpoints were trained on is prep v2. SLOTS_V3 exists in src now, and
# prepping with it here would feed every model pixels filed under a different table -- which
# would not crash, and would only show up as a lower score.
assert len(SLOTS) == 6, 'src slot table is not the six-slot layout these checkpoints trained on'
assert 'backbone_kwargs' in inspect.signature(KneeModel.__init__).parameters, (
    'src predates img_size passthrough, which the DINOv2 member needs')
assert 'slot_scheme' in inspect.signature(prep_slots).parameters, 'src predates prep v3'
print('src carries the Phase 6 prep and both heads |', len(CKPTS), 'checkpoints across',
      len(MEMBERS), 'members')

In [ ]:
import numpy as np
import pandas as pd
import torch

from knee.dataset import PreppedSlotDataset
from knee.infer import LABEL_COLUMNS, build_submission
from knee.prep import save_study_npz

# Exactly the training artifact's prep parameters and the confirmed config's read
# parameters. These are the numbers that must not drift from the confirm kernel.
PREP_GROUPS, PREP_CROP_MM, PREP_SIZE = 5, 130.0, 336
SLOT_NAMES = [n for n, _, _ in SLOTS]
# prep v2 explicitly: these checkpoints never saw the nineteen-slot table
PREP_SCHEME = 'v2'

test_df = pd.read_csv(f'{COMP_DIR}/test.csv')
test_series_df = pd.read_csv(f'{COMP_DIR}/test_series.csv')
TEST_ROOT = f'{COMP_DIR}/test_series'
print(f'{len(test_df)} test studies')

# Probe rather than assert: a hard GPU requirement turns an odd accelerator
# assignment into a zero on the scored run.
device = 'cpu'
if torch.cuda.is_available():
    try:
        _ = torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except Exception as exc:
        print(f'GPU present but unusable: {exc}')
print('device:', device)

for member in MEMBERS:
    # img_size only for the ViT: timm's efficientnet does not take it, and passing it
    # would raise here rather than at the first forward
    extra = {'img_size': member['read']} if member['backbone'].startswith('vit_') else {}
    member['slots'] = SLOT_NAMES[:member['n_slots']]
    member['models'] = []
    for path in member['ckpts']:
        m = KneeModel(backbone_name=member['backbone'], num_labels=len(LABEL_COLUMNS),
                      pretrained=False, head=member['head'], **extra)
        m.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        member['models'].append(m.to(device).eval())
    print(f"{member['name']}: {len(member['models'])} folds, {member['backbone']}, "
          f"{member['head']}, {member['n_slots']} slots at {member['read']}px")
n_models = sum(len(m['models']) for m in MEMBERS)
print(f'{n_models} models total')

In [ ]:
NPZ_DIR = tempfile.mkdtemp(prefix='knee_prep_')
prep_seconds, forward_seconds = [], []
lat_routes, slot_fill = {}, {}

def predict_one(study_uid):
    """ONE prep_slots -> npz -> each member reads it with its own slots and size ->
    probability mean over every model of every member. Anything raising in here
    becomes a row of 0.5 via build_submission."""
    t0 = time.time()
    slot_slices, meta = prep_slots(
        study_uid, TEST_ROOT, test_series_df,
        n_groups=PREP_GROUPS, crop_mm=PREP_CROP_MM, out_size=PREP_SIZE,
        slot_scheme=PREP_SCHEME)
    npz_path = os.path.join(NPZ_DIR, f'{study_uid}.npz')
    save_study_npz(npz_path, slot_slices, meta)
    prep_seconds.append(time.time() - t0)
    lat_routes[meta.get('route')] = lat_routes.get(meta.get('route'), 0) + 1
    slot_fill[len(slot_slices)] = slot_fill.get(len(slot_slices), 0) + 1

    try:
        t1 = time.time()
        per_model = []
        for member in MEMBERS:
            # a fresh read per member: the slot subset and read size are part of what
            # each member was trained on, and sharing one tensor would feed c3 six
            # slots it never saw
            dataset = PreppedSlotDataset([study_uid], NPZ_DIR, n_groups=PREP_GROUPS,
                                         out_size=member['read'], slots=member['slots'])
            image, mask, _, _ = dataset[0]
            batch = image.unsqueeze(0).to(device)
            mask_batch = mask.unsqueeze(0).to(device)
            with torch.no_grad():
                for m in member['models']:
                    per_model.append(torch.sigmoid(m(batch, mask=mask_batch))[0].cpu().numpy())
        # probability mean, not rank mean: measured 2026-09-07, prob-mean won 11 of 11
        # combinations on homogeneous members (NOTES, infer.rank_mean). Flat over all
        # ten models, so the two members carry equal weight at five folds each.
        probs = np.mean(per_model, axis=0)
        forward_seconds.append(time.time() - t1)
        return probs
    finally:
        os.remove(npz_path)

t0 = time.time()
submission = build_submission(test_df['StudyInstanceUID'].astype(str).tolist(), predict_one)
elapsed = time.time() - t0
print(f'{elapsed / 60:.1f} min for {len(submission)} studies '
      f'({elapsed / max(len(submission), 1):.2f} s/study)')
print(f'prep    {np.mean(prep_seconds):.3f}s/study (p95 {np.percentile(prep_seconds, 95):.3f})')
print(f'forward {np.mean(forward_seconds):.3f}s/study for {n_models} models '
      f'across {len(MEMBERS)} members')
print(f'laterality routes: {lat_routes}')
print(f'slots filled per study: {dict(sorted(slot_fill.items()))}')
fallbacks = int((submission[LABEL_COLUMNS] == 0.5).all(axis=1).sum())
print(f'0.5 fallbacks: {fallbacks} of {len(submission)}')

In [ ]:
submission.to_csv('submission.csv', index=False)
print(submission.head())

assert len(submission) == len(test_df), 'row count must match test studies exactly'
assert list(submission.columns) == ['StudyInstanceUID'] + LABEL_COLUMNS, 'column contract'
assert submission[LABEL_COLUMNS].isna().sum().sum() == 0, 'no NaNs allowed'
vals = submission[LABEL_COLUMNS].to_numpy()
assert (vals >= 0).all() and (vals <= 1).all(), 'probabilities must be in [0, 1]'

# Predictions should track the training positive rates and preserve their ordering.
# A wild departure means something broke upstream of scoring, which no assert above
# would catch.
print(f'\n{"label":20s} {"mean pred":>10s}')
for label in LABEL_COLUMNS:
    print(f'{label:20s} {submission[label].mean():10.4f}')